In [1]:
# Import libraries required for price elasticity analysis

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [2]:
# Load the feature-engineered pricing dataset

DATA_PATH = Path("../data/processed/pricing_features.parquet")

df = pd.read_parquet(DATA_PATH)

print("Dataset shape:", df.shape)
print("Date range:", df["date"].min(), "to", df["date"].max())

print("\nCategories:")
print(df["cat_id"].value_counts())

Dataset shape: (13563568, 30)
Date range: 2011-02-26 00:00:00 to 2016-05-22 00:00:00

Categories:
cat_id
FOODS        9425298
HOUSEHOLD    2693161
HOBBIES      1445109
Name: count, dtype: int64


In [3]:
# Aggregate daily observations to product-store-week level for elasticity estimation

weekly = (
    df.groupby(
        [
            "item_id",
            "dept_id",
            "cat_id",
            "store_id",
            "state_id",
            "wm_yr_wk"
        ],
        observed=True,
        as_index=False
    )
    .agg(
        sell_price=("sell_price", "mean"),
        weekly_demand=("demand", "sum"),
        date=("date", "min")
    )
)

weekly = weekly.sort_values(
    ["item_id", "store_id", "date"]
).reset_index(drop=True)

print("Daily rows:", len(df))
print("Weekly rows:", len(weekly))
print("Product-store series:", weekly.groupby(["item_id", "store_id"]).ngroups)

display(weekly.head())

Daily rows: 13563568
Weekly rows: 1943029
Product-store series: 7527


,item_id,dept_id,cat_id,store_id,state_id,wm_yr_wk,sell_price,weekly_demand,date
0,FOODS_1_001,FOODS_1,FOODS,CA_3,CA,11105,2.0,18,2011-02-26
1,FOODS_1_001,FOODS_1,FOODS,CA_3,CA,11106,2.0,19,2011-03-05
2,FOODS_1_001,FOODS_1,FOODS,CA_3,CA,11107,2.0,6,2011-03-12
3,FOODS_1_001,FOODS_1,FOODS,CA_3,CA,11108,2.0,8,2011-03-19
4,FOODS_1_001,FOODS_1,FOODS,CA_3,CA,11109,2.0,8,2011-03-26


In [4]:
# Identify historical price changes within each product-store series

group_cols = ["item_id", "store_id"]

weekly["previous_price"] = (
    weekly.groupby(group_cols, observed=True)["sell_price"]
    .shift(1)
)

weekly["previous_demand"] = (
    weekly.groupby(group_cols, observed=True)["weekly_demand"]
    .shift(1)
)

weekly["price_change_pct"] = (
    (weekly["sell_price"] - weekly["previous_price"])
    / weekly["previous_price"]
) * 100

weekly["demand_change_pct"] = (
    (weekly["weekly_demand"] - weekly["previous_demand"])
    / weekly["previous_demand"]
) * 100

price_change_weeks = weekly[
    weekly["previous_price"].notna() &
    (weekly["sell_price"] != weekly["previous_price"])
].copy()

print("Total weekly observations:", len(weekly))
print("Price-change weeks:", len(price_change_weeks))

display(
    price_change_weeks[
        [
            "date",
            "item_id",
            "store_id",
            "previous_price",
            "sell_price",
            "previous_demand",
            "weekly_demand",
            "price_change_pct",
            "demand_change_pct"
        ]
    ].head(10)
)

Total weekly observations: 1943029
Price-change weeks: 42812


,date,item_id,store_id,previous_price,sell_price,previous_demand,weekly_demand,price_change_pct,demand_change_pct
34,2011-10-22,FOODS_1_001,CA_3,2.00,1.75,0.0,2,-12.500000,inf
35,2011-10-29,FOODS_1_001,CA_3,1.75,2.00,2.0,11,14.285715,450.000000
79,2012-09-01,FOODS_1_001,CA_3,2.00,2.24,20.0,9,12.000000,-55.000000
141,2013-11-09,FOODS_1_001,CA_3,2.24,2.00,5.0,3,-10.714286,-40.000000
153,2014-02-01,FOODS_1_001,CA_3,2.00,2.24,0.0,18,12.000000,inf
322,2012-01-28,FOODS_1_001,TX_1,2.00,1.19,7.0,5,-40.500004,-28.571429
323,2012-02-04,FOODS_1_001,TX_1,1.19,2.00,5.0,3,68.067238,-40.000000
335,2012-04-28,FOODS_1_001,TX_1,2.00,1.00,6.0,7,-50.000000,16.666667
336,2012-05-05,FOODS_1_001,TX_1,1.00,2.00,7.0,6,100.000000,-14.285714
353,2012-09-01,FOODS_1_001,TX_1,2.00,2.24,11.0,9,12.000000,-18.181818


In [5]:
# Prepare positive price and demand observations for log-log elasticity estimation

elasticity_data = weekly[
    (weekly["sell_price"] > 0) &
    (weekly["weekly_demand"] > 0)
].copy()

elasticity_data["log_price"] = np.log(
    elasticity_data["sell_price"]
)

elasticity_data["log_demand"] = np.log(
    elasticity_data["weekly_demand"]
)

print("Weekly observations:", len(weekly))
print("Valid elasticity observations:", len(elasticity_data))
print(
    "Percentage retained:",
    round(len(elasticity_data) / len(weekly) * 100, 2),
    "%"
)

print("\nInfinite log values:")
print(
    np.isinf(
        elasticity_data[["log_price", "log_demand"]]
    ).sum()
)

Weekly observations: 1943029
Valid elasticity observations: 1629064
Percentage retained: 83.84 %

Infinite log values:
log_price     0
log_demand    0
dtype: int64


In [8]:
# Estimate log-log price elasticity separately for each product category

import statsmodels.api as sm

elasticity_results = []

for category, group in elasticity_data.groupby("cat_id", observed=True):

    X = sm.add_constant(group["log_price"])
    y = group["log_demand"]

    model = sm.OLS(y, X).fit()

    elasticity_results.append({
        "category": category,
        "elasticity": model.params["log_price"],
        "r_squared": model.rsquared,
        "observations": len(group)
    })

category_elasticity = pd.DataFrame(elasticity_results)

display(
    category_elasticity.sort_values("elasticity")
)

,category,elasticity,r_squared,observations
0,FOODS,-0.632842,0.103975,1124297
2,HOUSEHOLD,-0.534156,0.074675,327473
1,HOBBIES,-0.453222,0.274292,177294


In [9]:
# Estimate price elasticity within each product-store series using log-log regression

product_elasticity_results = []

for (item_id, store_id), group in elasticity_data.groupby(
    ["item_id", "store_id"],
    observed=True
):
    if group["sell_price"].nunique() < 3:
        continue

    X = sm.add_constant(group["log_price"])
    y = group["log_demand"]

    model = sm.OLS(y, X).fit()

    product_elasticity_results.append({
        "item_id": item_id,
        "store_id": store_id,
        "cat_id": group["cat_id"].iloc[0],
        "elasticity": model.params["log_price"],
        "r_squared": model.rsquared,
        "observations": len(group),
        "unique_prices": group["sell_price"].nunique()
    })

product_elasticity = pd.DataFrame(product_elasticity_results)

print("Estimated product-store elasticities:", len(product_elasticity))

display(product_elasticity.head(10))

Estimated product-store elasticities: 7355


,item_id,store_id,cat_id,elasticity,r_squared,observations,unique_prices
0,FOODS_1_001,CA_3,FOODS,-3.472650,0.045996,247,3
1,FOODS_1_001,TX_1,FOODS,-0.819400,0.011198,229,6
2,FOODS_1_002,CA_1,FOODS,-2.923871,0.076821,216,4
3,FOODS_1_002,CA_2,FOODS,-1.854139,0.018825,216,4
4,FOODS_1_002,CA_4,FOODS,0.163031,0.000221,235,3
5,FOODS_1_002,WI_1,FOODS,3.764040,0.134128,267,3
6,FOODS_1_003,TX_1,FOODS,-3.240972,0.054110,244,3
7,FOODS_1_004,CA_1,FOODS,-1.571080,0.016338,200,3
8,FOODS_1_004,CA_2,FOODS,-7.282288,0.260577,202,3
9,FOODS_1_004,CA_3,FOODS,1.323192,0.008146,191,3


In [10]:
# Inspect the distribution and reliability of product-store elasticity estimates

print("Elasticity distribution:")
display(product_elasticity["elasticity"].describe())

print("\nR-squared distribution:")
display(product_elasticity["r_squared"].describe())

print(
    "\nNegative elasticity:",
    round((product_elasticity["elasticity"] < 0).mean() * 100, 2),
    "%"
)

print(
    "Positive elasticity:",
    round((product_elasticity["elasticity"] > 0).mean() * 100, 2),
    "%"
)

print(
    "Elasticity between -5 and 0:",
    round(
        product_elasticity["elasticity"].between(-5, 0).mean() * 100,
        2
    ),
    "%"
)

Elasticity distribution:


count    7355.000000
mean       -0.506809
std         4.030036
min       -19.863992
25%        -2.546631
50%        -0.829760
75%         0.864493
max        38.289054
Name: elasticity, dtype: float64


R-squared distribution:


count    7.355000e+03
mean     7.410324e-02
std      1.056765e-01
min      2.779397e-10
25%      6.006474e-03
50%      2.995118e-02
75%      9.800480e-02
max      8.915903e-01
Name: r_squared, dtype: float64


Negative elasticity: 63.68 %
Positive elasticity: 36.32 %
Elasticity between -5 and 0: 55.72 %


In [11]:
# Filter product-store elasticities to retain interpretable and sufficiently supported estimates

reliable_elasticity = product_elasticity[
    (product_elasticity["elasticity"] < 0) &
    (product_elasticity["elasticity"] >= -5) &
    (product_elasticity["observations"] >= 100) &
    (product_elasticity["unique_prices"] >= 3) &
    (product_elasticity["r_squared"] >= 0.05)
].copy()

print("Total estimates:", len(product_elasticity))
print("Reliable estimates:", len(reliable_elasticity))
print(
    "Percentage retained:",
    round(len(reliable_elasticity) / len(product_elasticity) * 100, 2),
    "%"
)

display(
    reliable_elasticity["elasticity"].describe()
)

Total estimates: 7355
Reliable estimates: 1486
Percentage retained: 20.2 %


count    1486.000000
mean       -2.945628
std         1.029039
min        -4.997842
25%        -3.747626
50%        -2.897694
75%        -2.158161
max        -0.666016
Name: elasticity, dtype: float64

In [12]:
# Create category-level median elasticities as fallbacks for products without reliable estimates

category_fallback = (
    reliable_elasticity
    .groupby("cat_id", observed=True)["elasticity"]
    .median()
    .reset_index(name="fallback_elasticity")
)

overall_fallback = reliable_elasticity["elasticity"].median()

print("Category fallback elasticities:")
display(category_fallback)

print(
    "\nOverall fallback elasticity:",
    round(overall_fallback, 3)
)

Category fallback elasticities:


,cat_id,fallback_elasticity
0,FOODS,-2.897328
1,HOBBIES,-3.615260
2,HOUSEHOLD,-2.724296



Overall fallback elasticity: -2.898


In [13]:
# Create the final elasticity lookup using product-store estimates with category and overall fallbacks

all_series = (
    df[
        ["item_id", "store_id", "cat_id"]
    ]
    .drop_duplicates()
    .copy()
)

final_elasticity = all_series.merge(
    reliable_elasticity[
        ["item_id", "store_id", "elasticity", "r_squared"]
    ],
    on=["item_id", "store_id"],
    how="left"
)

final_elasticity = final_elasticity.merge(
    category_fallback,
    on="cat_id",
    how="left"
)

final_elasticity["elasticity_source"] = np.where(
    final_elasticity["elasticity"].notna(),
    "product_store",
    "category_fallback"
)

final_elasticity["final_elasticity"] = (
    final_elasticity["elasticity"]
    .fillna(final_elasticity["fallback_elasticity"])
    .fillna(overall_fallback)
)

print("Total product-store series:", len(final_elasticity))

print("\nElasticity source:")
print(final_elasticity["elasticity_source"].value_counts())

print("\nMissing final elasticities:")
print(final_elasticity["final_elasticity"].isna().sum())

display(final_elasticity.head())

Total product-store series: 7527

Elasticity source:
elasticity_source
category_fallback    6041
product_store        1486
Name: count, dtype: int64

Missing final elasticities:
0


,item_id,store_id,cat_id,elasticity,r_squared,fallback_elasticity,elasticity_source,final_elasticity
0,FOODS_1_001,CA_3,FOODS,NaN,NaN,-2.897328,category_fallback,-2.897328
1,FOODS_1_001,TX_1,FOODS,NaN,NaN,-2.897328,category_fallback,-2.897328
2,FOODS_1_002,CA_1,FOODS,-2.923871,0.076821,-2.897328,product_store,-2.923871
3,FOODS_1_002,CA_2,FOODS,NaN,NaN,-2.897328,category_fallback,-2.897328
4,FOODS_1_002,CA_4,FOODS,NaN,NaN,-2.897328,category_fallback,-2.897328


In [14]:
# Validate the final elasticity values that will be used by the pricing simulator

print("Final elasticity distribution:")
display(final_elasticity["final_elasticity"].describe())

print("\nElasticity by source:")
display(
    final_elasticity.groupby("elasticity_source")["final_elasticity"]
    .agg(["count", "mean", "median", "min", "max"])
)

print(
    "\nAll series covered:",
    final_elasticity["final_elasticity"].notna().all()
)

Final elasticity distribution:


count    7527.000000
mean       -2.944590
std         0.510450
min        -4.997842
25%        -2.897328
50%        -2.897328
75%        -2.724296
max        -0.666016
Name: final_elasticity, dtype: float64


Elasticity by source:


,count,mean,median,min,max
elasticity_source,,,,,
category_fallback,6041,-2.944335,-2.897328,-3.615260,-2.724296
product_store,1486,-2.945628,-2.897694,-4.997842,-0.666016



All series covered: True


In [16]:
# Save the final elasticity lookup for the pricing simulator

from pathlib import Path

PROCESSED_DATA_DIR = Path("../data/processed")
output_path = PROCESSED_DATA_DIR / "elasticity_lookup.parquet"

final_elasticity[
    [
        "item_id",
        "store_id",
        "cat_id",
        "final_elasticity",
        "elasticity_source",
        "r_squared"
    ]
].to_parquet(
    output_path,
    index=False,
    engine="pyarrow"
)

print("Saved to:", output_path)
print("Rows saved:", len(final_elasticity))
print("Missing elasticities:", final_elasticity["final_elasticity"].isna().sum())

Saved to: ..\data\processed\elasticity_lookup.parquet
Rows saved: 7527
Missing elasticities: 0


In [17]:
# Display the final summary of the price elasticity analysis

print("PRICE ELASTICITY ANALYSIS COMPLETE")
print("-" * 40)

print(f"Total product-store series: {len(final_elasticity)}")
print(f"Reliable product-store estimates: {(final_elasticity['elasticity_source'] == 'product_store').sum()}")
print(f"Category fallback estimates: {(final_elasticity['elasticity_source'] == 'category_fallback').sum()}")
print(f"Missing elasticities: {final_elasticity['final_elasticity'].isna().sum()}")

print("\nCategory fallback elasticities:")
display(category_fallback)

print(f"Overall fallback elasticity: {overall_fallback:.3f}")

PRICE ELASTICITY ANALYSIS COMPLETE
----------------------------------------
Total product-store series: 7527
Reliable product-store estimates: 1486
Category fallback estimates: 6041
Missing elasticities: 0

Category fallback elasticities:


,cat_id,fallback_elasticity
0,FOODS,-2.897328
1,HOBBIES,-3.615260
2,HOUSEHOLD,-2.724296


Overall fallback elasticity: -2.898
